**Data driven CT prediction model**

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import optuna
from lightgbm import LGBMRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV, KFold,cross_val_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, AdaBoostRegressor
from sklearn.tree import DecisionTreeRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score,mean_absolute_error,make_scorer
from sklearn.feature_selection import mutual_info_regression
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LassoCV
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_regression, SelectKBest, SequentialFeatureSelector
from sklearn.linear_model import LinearRegression
from scipy.optimize import minimize
from sklearn.linear_model import Ridge
from sklearn.utils import resample
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib.pyplot as plt
import seaborn as sns
import random
import shap
import os
import joblib

#from minisom import MiniSom

def train_with_kfold(model, X_subset, y_train, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    scores = []
    models = []

    for train_index, val_index in kf.split(X_subset):
        X_train_fold, X_val_fold = X_subset.iloc[train_index], X_subset.iloc[val_index]
        y_train_fold, y_val_fold = y_train[train_index], y_train[val_index]

        model.fit(X_train_fold, y_train_fold)
        score = model.score(X_val_fold, y_val_fold)  # R^2 score
        scores.append(score)
        models.append(model)

    return np.mean(scores), models

def random_forest_model(X_train, y_train):
    """
    Random Forest model with grid search for hyperparameters.
    """
    # Define the model and hyperparameter grid
    model = RandomForestRegressor(random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200, 400, 600, 1000],
        'max_depth': [1, 3, 5, 7, 10],
        'min_samples_split': [2, 5, 10]
    }

    # Perform grid search with k-fold cross-validation
    #grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='r2', cv=5)
    grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, scoring='r2', cv=5, n_iter=50, random_state=42)


    grid_search.fit(X_train, y_train)

    # Get the best model and save it
    best_model = grid_search.best_estimator_
    print("Best Random Forest Hyperparameters:", grid_search.best_params_)
    model_name=input("Set the RF model name:")#"best_rf_model_6features_newdataset.pkl"
    save_dir = "/content/drive/My Drive/Colab Notebooks/Ozonation model/"
    save_path = os.path.join(save_dir, model_name)
    joblib.dump(best_model, save_path)
    print(f"✅ Best Random Forest model saved to: {save_path}")
    return best_model

def multi_random_forest_model(X_train, y_train, selected_features, X_test, num_models=15, random_state=42):
    """
    Trains multiple Random Forest models, each with a subset of features,
    and combines their predictions using weighted averaging.

    Returns:
        y_pred_final: Final weighted prediction array
        best_models: List of tuples (model, features)
        best_scores: List of R² scores for each model
    """
    np.random.seed(random_state)
    best_models = []
    best_scores = []
    used_feature_sets = set()

    param_grid = {
        'n_estimators': [50, 100, 200, 400, 600, 1000],
        'max_depth': [1, 3, 5, 7, 10],
        'min_samples_split': [2, 5, 7, 10]
    }

    while len(best_models) < num_models:
        # Randomly select a subset of features
        num_features = np.random.randint(3, len(selected_features) + 1)
        features = np.random.choice(selected_features, size=num_features, replace=False).tolist()
        features_tuple = tuple(sorted(features))

        if features_tuple in used_feature_sets:
            continue
        used_feature_sets.add(features_tuple)

        print(f"Training model {len(best_models)+1}/{num_models} with features: {features}")
        X_subset = X_train[features]
        model = RandomForestRegressor(random_state=random_state)
        grid_search = GridSearchCV(model, param_grid, cv=5, scoring='r2', n_jobs=-1)
        grid_search.fit(X_subset, y_train)

        best_model = grid_search.best_estimator_
        best_score = grid_search.best_score_

        best_models.append((best_model, features))
        best_scores.append(best_score)

    # Filter models with R² >= 0.5
    mask = best_scores >= 0.5
    best_models = [m for i, m in enumerate(best_models) if mask[i]]
    best_scores = best_scores[mask]

    if len(best_models) == 0:
        raise ValueError("No RF models achieved R² >= 0.6")

    # Normalize weights
    eps = 1e-6
    weights = np.array(best_scores) / (np.sum(best_scores) + eps)

    # Weighted predictions
    y_preds = np.array([
        model.predict(X_test[features]) for model, features in best_models
    ])
    y_pred_rf = np.average(y_preds, axis=0, weights=weights)

    # Print top 3 models and their selected features
    print("\nTop 3 models based on R² score:")
    top_indices = np.argsort(best_scores)[-3:][::-1]  # Indices of top 3 scores
    for i in top_indices:
        print(f"Model {i+1}: R² = {best_scores[i]:.4f}, Features = {best_models[i][1]}")

    best_model = best_models
    return y_pred_rf, best_model, best_scores

def xgboost_model(X_train, y_train, search_method="bayesian"):
    """
    Trains an XGBoost model using specified hyperparameter search method.

    Parameters:
        X_train, y_train: Training data
        search_method: 'grid', 'random', or 'bayesian'

    Returns:
        Trained model with best hyperparameters
    """

    model = XGBRegressor(random_state=42)

    param_grid = {
        'n_estimators': [50, 100, 200, 400, 600, 1000],
        'max_depth': [1, 3, 5, 7, 10],
        'learning_rate': [0.01, 0.05,  0.1, 0.2, 0.5]
    }

    if search_method == "grid":
        print("Running Grid Search...")
        grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='r2', cv=5)
        grid_search.fit(X_train, y_train)
        best_model = grid_search.best_estimator_
        print("Best Grid Search Hyperparameters:", grid_search.best_params_)

    elif search_method == "random":
        print("Running Random Search...")
        random_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, scoring='r2', cv=5, n_iter=50, random_state=42)
        random_search.fit(X_train, y_train)
        best_model = random_search.best_estimator_
        print("Best Random Search Hyperparameters:", random_search.best_params_)

    elif search_method == "bayesian":
        print("Running Bayesian Optimization with Optuna...")

        def objective(trial):
            params = {
                'n_estimators': trial.suggest_int('n_estimators', 50, 1000),
                'max_depth': trial.suggest_int('max_depth', 1, 10),
                'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.5, log=True),
                'random_state': 42
            }
            model = XGBRegressor(**params)
            scores = cross_val_score(model, X_train, y_train, scoring='r2', cv=5)
            return scores.mean()

        study = optuna.create_study(direction='maximize')
        study.optimize(objective, n_trials=50, show_progress_bar=True)

        print("Best Bayesian Optimization Hyperparameters:", study.best_params)

        best_model = XGBRegressor(**study.best_params)
        best_model.fit(X_train, y_train)

    else:
        raise ValueError("Invalid search_method. Choose from 'grid', 'random', or 'bayesian'.")

    # Get the best model and save it
    model_name=input("Set the XGBoost model name:")#"xgb_model_7features_newdataset_bestpervariable.pkl"
    save_dir = "/content/drive/My Drive/Colab Notebooks/Ozonation model/"
    save_path = os.path.join(save_dir, model_name)
    joblib.dump(best_model, save_path)
    print(f"✅ Best XGBoosting model saved to: {save_path}")

    return best_model

def multi_xgboost_model(X_train, y_train, selected_features, X_test, num_models=15, random_state=42):
    """
    Trains multiple XGBoost models with random feature subsets and returns
    the weighted average of their predictions, plus their details.
    """
    np.random.seed(random_state)
    best_models = []
    best_scores = []
    used_feature_sets = set()

    param_grid = {
        'n_estimators': [50, 100, 200, 400, 600, 1000],
        'max_depth': [1, 3, 5, 7, 10],
        'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.5]
    }

    while len(best_models) < num_models:
        # Randomly select a subset of features
        num_features = np.random.randint(3, len(selected_features) + 1)
        features = np.random.choice(selected_features, size=num_features, replace=False).tolist()
        features_tuple = tuple(sorted(features))

        if features_tuple in used_feature_sets:
            continue
        used_feature_sets.add(features_tuple)

        print(f"Training XGBoost model {len(best_models)+1}/{num_models} with features: {features}")
        X_subset = X_train[features]
        model = XGBRegressor(random_state=random_state, verbosity=0)
        grid_search = GridSearchCV(model, param_grid, cv=5, scoring='r2', n_jobs=-1)
        grid_search.fit(X_subset, y_train)

        best_model = grid_search.best_estimator_
        best_score = grid_search.best_score_

        best_models.append((best_model, features))
        best_scores.append(best_score)

    # Filter models with R² >= 0.5
    mask = best_scores >= 0.5
    best_models = [m for i, m in enumerate(best_models) if mask[i]]
    best_scores = best_scores[mask]

    if len(best_models) == 0:
        raise ValueError("No RF models achieved R² >= 0.6")

    # Normalize weights
    eps = 1e-6
    weights = np.array(best_scores) / (np.sum(best_scores) + eps)

    # Weighted predictions
    y_preds = np.array([
        model.predict(X_test[features]) for model, features in best_models
    ])
    y_pred_xgb = np.average(y_preds, axis=0, weights=weights)

    # Print top 3 models and their selected features
    print("\nTop 3 XGBoost models based on R² score:")
    top_indices = np.argsort(best_scores)[-3:][::-1]
    for i in top_indices:
        print(f"Model {i+1}: R² = {best_scores[i]:.4f}, Features = {best_models[i][1]}")

    best_model = best_models
    return y_pred_xgb, best_model, best_scores

def combined_rf_xgb_ensemble(best_models_rf, best_scores_rf, best_models_xgb, best_scores_xgb, X_test):
    """
    Combines the best RF and XGBoost models into a weighted ensemble.

    Parameters:
    - best_models_rf: list of tuples (model, features) for RF
    - best_scores_rf: list of R² scores for RF models
    - best_models_xgb: list of tuples (model, features) for XGB
    - best_scores_xgb: list of R² scores for XGB models
    - X_test: test input dataframe

    Returns:
    - y_pred_final: final ensemble prediction
    """
    # Combine models and scores
    combined_models = best_models_rf + best_models_xgb
    combined_scores = best_scores_rf + best_scores_xgb

    # Normalize weights
    eps = 1e-6
    weights = np.array(combined_scores) / (np.sum(combined_scores) + eps)

    # Predict from all models
    y_preds = np.array([
        model.predict(X_test[features]) for model, features in combined_models
    ])

    # Weighted average
    y_pred_final = np.average(y_preds, axis=0, weights=weights)

    return y_pred_final,combined_models

def top_features_random_forest_model(X_train, y_train, input_features, X_test, y_test,models_info):

    best_models = {}
    best_scores = {}
    y_preds = {}
    for method, features in models_info.items():
        print(f"\n🔹 {method}:")
        print(f"Selected Features ({len(features)}): {features}")
        X_train_sub = X_train[features]
        X_test_sub = X_test[features]
        model = RandomForestRegressor(random_state=42)
        param_grid = {
             'n_estimators': [50, 100, 200, 400, 600, 1000],
             'max_depth': [1, 3, 5, 7, 10],
             'min_samples_split': [2, 5, 10]
        }

    # Perform grid search with k-fold cross-validation
    #grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='r2', cv=5)
        grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, scoring='r2', cv=5, n_iter=50, random_state=42)
        grid_search.fit(X_train_sub, y_train)

    # Get the best model
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test_sub)
        test_score = r2_score(y_test, y_pred)
        print("Best Random Forest Hyperparameters:", grid_search.best_params_)
        print(f"Test R² Score: {test_score:.4f}")
        best_models[method] = best_model
        best_scores[method] = test_score
        y_preds[method] = y_pred
    preds_array = np.column_stack(list(y_preds.values()))
    # Normalize weights
    eps = 1e-6
    score_values = np.array(list(best_scores.values()))
    weights = np.array(score_values) / (np.sum(score_values) + eps)
    y_pred_final = np.average(preds_array, axis=1, weights=weights)
    return best_models, y_pred_final

def top_features_xgb_model(X_train, y_train, input_features, X_test, y_test,models_info):

    best_models = {}
    best_scores = {}
    y_preds = {}
    for method, features in models_info.items():
        print(f"\n🔹 {method}:")
        print(f"Selected Features ({len(features)}): {features}")
        X_train_sub = X_train[features]
        X_test_sub = X_test[features]
        model = XGBRegressor(random_state=42)
        param_grid = {
             'n_estimators': [50, 100, 200, 400, 600, 1000],
             'max_depth': [1, 3, 5, 7, 10],
             'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.5]
        }

    # Perform grid search with k-fold cross-validation
    #grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='r2', cv=5)
        grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, scoring='r2', cv=5, n_iter=50, random_state=42)
        grid_search.fit(X_train_sub, y_train)

    # Get the best model
        best_model = grid_search.best_estimator_
        y_pred = best_model.predict(X_test_sub)
        test_score = r2_score(y_test, y_pred)
        print("Best XGB Hyperparameters:", grid_search.best_params_)
        print(f"Test R² Score: {test_score:.4f}")
        best_models[method] = best_model
        best_scores[method] = test_score
        y_preds[method] = y_pred
    preds_array = np.column_stack(list(y_preds.values()))
    # Normalize weights
    eps = 1e-6
    score_values = np.array(list(best_scores.values()))
    weights = np.array(score_values) / (np.sum(score_values) + eps)
    y_pred_final = np.average(preds_array, axis=1, weights=weights)
    return best_models, y_pred_final

def gb_model(X_train, y_train):
    """
    Gradient boosting model with grid search for hyperparameters.
    """
    # Define the model and hyperparameter grid
    model = GradientBoostingRegressor(random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200, 400, 600, 1000],
        'max_depth': [1, 3, 5, 7, 10],
        'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.5]
    }

    # Perform grid search with k-fold cross-validation
    #grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='r2', cv=5)
    grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, scoring='r2', cv=5, n_iter=50, random_state=42)


    grid_search.fit(X_train, y_train)

    # Get the best model
    best_model = grid_search.best_estimator_
    print("Best Gradient Boosting Hyperparameters:", grid_search.best_params_)

    # Get the best model and save it
    model_name=input("Set the Gradient Boosting model name:")#"best_gb_model_6features_newdataset.pkl"
    save_dir = "/content/drive/My Drive/Colab Notebooks/Ozonation model/"
    save_path = os.path.join(save_dir, model_name)
    joblib.dump(best_model, save_path)
    print(f"✅ Best Gradient Boosting model saved to: {save_path}")
    return best_model

def adaboost_model(X_train, y_train):
    """
    Adaboost model with grid search for hyperparameters.
    """
    # Define the model and hyperparameter grid
    base_estimator = DecisionTreeRegressor()
    model = AdaBoostRegressor(estimator=base_estimator, random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200, 400, 600, 1000],
        'estimator__max_depth': [1, 3, 5, 7, 10],
        'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.5]
    }

    # Perform grid search with k-fold cross-validation
    #grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='r2', cv=5)
    grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, scoring='r2', cv=5, n_iter=50, random_state=42,n_jobs=-1)
    grid_search.fit(X_train, y_train)

    # Get the best model
    best_model = grid_search.best_estimator_
    print("Best AdaBoosting Hyperparameters:", grid_search.best_params_)
    model_name=input("Set the AdaBoost model name:")#"best_ada_model_13features_bromate.pkl"
    save_dir = "/content/drive/My Drive/Colab Notebooks/Ozonation model/"
    save_path = os.path.join(save_dir, model_name)
    joblib.dump(best_model, save_path)
    print(f"✅ Best AdaBoost model saved to: {save_path}")

    return best_model

def lgb_model(X_train, y_train):
    """
    Light GB model with grid search for hyperparameters.
    """
    # Define the model and hyperparameter grid
    model = LGBMRegressor(random_state=42)
    param_grid = {
        'n_estimators': [50, 100, 200, 400, 600, 1000],
        'max_depth': [1, 3, 5, 7, 10],
        'learning_rate': [0.01, 0.05, 0.1, 0.2, 0.5],
        'num_leaves': [15, 31, 63, 127]  # optional: more control over complexity
    }

    # Perform grid search with k-fold cross-validation
    #grid_search = GridSearchCV(estimator=model, param_grid=param_grid, scoring='r2', cv=5)
    grid_search = RandomizedSearchCV(estimator=model, param_distributions=param_grid, scoring='r2', cv=5, n_iter=50, random_state=42,n_jobs=-1)
    grid_search.fit(X_train, y_train)

    # Get the best model
    best_model = grid_search.best_estimator_
    print("Best Light Gradient Boosting Hyperparameters:", grid_search.best_params_)
    model_name=input("Set the Light Boosting model name:")#"best_gb_model_6features_newdataset.pkl"
    save_dir = "/content/drive/My Drive/Colab Notebooks/Ozonation model/"
    save_path = os.path.join(save_dir, model_name)
    joblib.dump(best_model, save_path)
    print(f"✅ Best Light Boosting model saved to: {save_path}")
    return best_model

def run_multi_model_ensemble(X_train, y_train, X_test, method="weighted", k=10, random_state=42):
    """
    Run RF, GB, XGB models using randomized search + KFold CV.
    Select models with R² >= 0.6 and combine them using the chosen ensemble method.

    Parameters:
    - X_train, y_train: training data
    - X_test: test data
    - method: 'weighted' | 'optimised' | 'stacking'
    - k: number of folds
    - random_state: reproducibility

    Returns:
    - y_pred_final: ensemble prediction
    - selected_models: list of (best_model, features)
    - selected_scores: list of R² scores
    """
    np.random.seed(random_state)
    models_info = {
        "rf": {
            "estimator": RandomForestRegressor(random_state=random_state),
            "param_distributions": {
                "n_estimators": [100, 200, 400, 600],
                "max_depth": [3, 5, 7, 10, None],
                "min_samples_split": [2, 5, 10],
                "min_samples_leaf": [1, 2, 4]
            }
        },
        "gb": {
            "estimator": GradientBoostingRegressor(random_state=random_state),
            "param_distributions": {
                "n_estimators": [100, 200, 400],
                "max_depth": [3, 5, 7],
                "learning_rate": [0.01, 0.05, 0.1, 0.2]
            }
        },
        "xgb": {
            "estimator": XGBRegressor(random_state=random_state, verbosity=0),
            "param_distributions": {
                "n_estimators": [100, 200, 400],
                "max_depth": [3, 5, 7],
                "learning_rate": [0.01, 0.05, 0.1, 0.2],
                "subsample": [0.7, 0.8, 1.0],
                "colsample_bytree": [0.7, 0.8, 1.0]
            }
        }
    }

    kf = KFold(n_splits=k, shuffle=True, random_state=random_state)
    selected_models = []
    selected_scores = []

    for name, info in models_info.items():
        print(f"\nRunning {name.upper()} with RandomizedSearchCV + {k}-Fold CV")
        model = info["estimator"]
        param_dist = info["param_distributions"]

        search = RandomizedSearchCV(
            estimator=model,
            param_distributions=param_dist,
            n_iter=10,  # you can increase this
            cv=kf,
            scoring=make_scorer(r2_score),
            n_jobs=-1,
            random_state=random_state
        )
        search.fit(X_train, y_train)

        best_model = search.best_estimator_
        best_score = search.best_score_
        print(f"{name.upper()} best CV R² = {best_score:.3f}")

        # Only keep if model is decent
        if best_score >= 0.5:
            selected_models.append((best_model, X_train.columns.tolist()))
            selected_scores.append(best_score)

    if not selected_models:
        raise ValueError("No models passed the R² threshold of 0.5!")

    # --- Ensemble step ---
    y_pred_final, weights, meta_model = generalized_ensemble(
        selected_models, selected_scores, X_test,
        method=method,
        y_valid=y_train if method != "weighted" else None
    )

    return y_pred_final, selected_models, selected_scores


def generalized_ensemble(models, scores, X_test, method, y_valid=None, meta_learner=None):
    """
    Generalized ensemble for RF, XGB, LGBM, GBM models.


    Parameters:
    - models: list of (model, features)
    - scores: list of validation R² scores
    - X_test: dataframe for predictions
    - method: "weighted" | "optimized" | "stacking"
    - y_valid: true y values for validation (required for optimized/stacking)
    - meta_learner: meta model for stacking (default = Ridge)


    Returns:
    - y_pred_final: final ensemble prediction
    """


    # Predictions from all base models
    preds = np.column_stack([
    model.predict(X_test[features]) for model, features in models])


    if method == "weighted":
    # Simple normalized weighting by scores
        eps = 1e-6
        weights = np.array(scores) / (np.sum(scores) + eps)
        y_pred_final = np.average(preds, axis=1, weights=weights)
        return y_pred_final


    elif method == "optimised":
         if y_valid is None:
             raise ValueError("y_valid is required for optimized ensemble")


         def objective(weights):
             weights = np.array(weights)
             weights = weights / np.sum(weights)
             y_pred = np.dot(preds, weights)
             return mean_squared_error(y_valid, y_pred)


         # Initial weights
         init_weights = np.ones(len(models)) / len(models)


         # Constraints: sum(weights) = 1, weights >= 0
         constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})
         bounds = [(0, 1)] * len(models)


         result = minimize(objective, init_weights, bounds=bounds, constraints=constraints)
         opt_weights = result.x / np.sum(result.x)


         y_pred_final = np.dot(preds, opt_weights)
         return y_pred_final


    elif method == "stacking":
        if y_valid is None:
            raise ValueError("y_valid is required for stacking ensemble")


        # Use Ridge as default meta-learner if not provided
        if meta_learner is None:
            meta_learner = Ridge(alpha=1.0)


        # Fit meta-learner on validation set
        meta_learner.fit(preds, y_valid)
        y_pred_final = meta_learner.predict(preds)
        return y_pred_final
    else:
        raise ValueError("Invalid method. Choose from 'weighted', 'optimized', 'stacking'.")

def physics_informed_mlp(X_train_ps, X_test_ps, y_train_ps, Ko3, k_UVA, CO3_t_train, CO3_t_test,T_train,T_test):
    """
    Physics-Informed MLP model to predict ozone (Co3) using one of two ODEs:
    1. dC/dt = -K_o3 * C
    2. dC/dt = -K_o3 * C - K_uva * (UVA - UVAo) * Y
    """

    class OzoneMLP(nn.Module):
        def __init__(self, input_size, hidden_size):
            super(OzoneMLP, self).__init__()
        # 1. Shared Feature Extractor (Temp, Turbidity, Flow, Dosage)
            self.feature_extractor = nn.Sequential(
                nn.Linear(input_size, hidden_size),
                nn.Tanh(),
                nn.Linear(hidden_size, hidden_size),
                nn.Tanh()
            )

        # 2. Head for Kinetic Constant (Ko3) - Depends only on water quality
            self.ko3_head = nn.Sequential(
                nn.Linear(hidden_size, 1),
                nn.Softplus() # Ensures decay rate is always positive
            )

        # 3. Head for Ozone Concentration - Depends on features + Time
            self.o3_head = nn.Sequential(
                nn.Linear(hidden_size + 1, hidden_size),
                nn.Tanh(),
                nn.Linear(hidden_size, 1),
                nn.Softplus() # Ensures concentration is always positive
            )

        def forward(self, x, t):
            # Extract features from Temp, Turbidity, Flow, Dosage
            features = self.feature_extractor(x)

            # Predict Ko3 (The 'Physics' parameter)
            ko3 = self.ko3_head(features)

            # Predict Ozone at time t
            combined_input = torch.cat([features, t], dim=1)
            o3_val = self.o3_head(combined_input)

            return o3_val, ko3

    # Set seed for reproducibility
    torch.manual_seed(42)

    # Hyperparameters
    input_size = X_train_ps.shape[1]
    hidden_size = int(input("Set the size of the hidden layer (16,32,64,128,256):"))
    n_epochs = int(input("Set the number of epochs:"))
    lambda_data = float(input("Set the weight of the data (MSE) losses (0-1)"))
    lambda_phys = float(input("Set the weight of the physics informed losses (0-1)"))

    # Data prep
    num_samples = X_train_ps.shape[0]
    num_steps = CO3_t_train.shape[1]
    t_cols = [f'T_{i}' for i in range(2, 2 + num_steps)]
    o3_cols = [f'O3_{i}' for i in range(2, 2 + num_steps)]


    # 1. Flatten the Time and Ozone values
    T_flattened = torch.tensor(T_train[t_cols].values, dtype=torch.float32).reshape(-1, 1).requires_grad_(True)
    Co3_true = torch.tensor(CO3_t_train[o3_cols].values, dtype=torch.float32).reshape(-1, 1)

    # 2. Expand static features to match flattened time
    X_static = torch.tensor(X_train_ps.values, dtype=torch.float32)
    X_expanded = X_static.repeat_interleave(num_steps, dim=0)

    # 3. Initial Condition Data (t=0)
    dosages = torch.tensor(X_train_ps["Ozonedosage"].values, dtype=torch.float32).view(-1, 1)
    t0 = torch.zeros((X_train_ps.shape[0], 1), dtype=torch.float32).requires_grad_(True)

    # --- Training ---
    pinn = OzoneMLP(input_size, hidden_size)
    optimizer = torch.optim.Adam(pinn.parameters(), lr=1e-3)
    criterion = nn.MSELoss() # Instantiate the loss function**

    for epoch in range(n_epochs):
        pinn.train()
        optimizer.zero_grad()

        # 1. FORWARD PASS
        Co3_pred, ko3_pred = pinn(X_expanded,T_flattened)
        #print(Co3_pred)

        # 2. DATA LOSS (MSE against measurements)
        loss_data = criterion(Co3_pred, Co3_true)

        # 3. PHYSICS LOSS (The ODE Residual)
        dO3_dt = torch.autograd.grad(Co3_pred, T_flattened,
                                     grad_outputs=torch.ones_like(Co3_pred),
                                     create_graph=True)[0]

        physics_residual = dO3_dt + ko3_pred * Co3_pred
        loss_phys = (physics_residual**2).mean()

        # 4. INITIAL CONDITION LOSS (O3 at t=0 must equal Dosage)
        o3_t0, _ = pinn(X_static, t0)
        loss_ic = criterion(o3_t0, dosages)

        # 5. TOTAL LOSS
        # Weighting: IC and Physics are often more important initially
        total_loss = (lambda_data * loss_data) + (lambda_phys * loss_phys) + (0 * loss_ic)

        total_loss.backward()
        optimizer.step()

        # Echo
        if epoch % 1000 == 0:
            print(f"Epoch {epoch} | Total: {total_loss.item():.5f} |Data: {loss_data.item():.5f} | Phys: {loss_phys.item():.5f} | IC: {loss_ic.item():.5f}")

    # --- Testing / Prediction ---
    pinn.eval()
    num_samples_test = X_test_ps.shape[0]
    num_steps_test = CO3_t_test.shape[1]


    # Convert predicting inputs to tensors
    X_test_static = torch.tensor(X_test_ps.values, dtype=torch.float32)
    X_test_expanded = X_test_static.repeat_interleave(num_steps_test, dim=0)
    T_test_flattened = torch.tensor(T_test[t_cols].values, dtype=torch.float32).reshape(-1, 1)

    # Ground truth Co3_test in the predicting inputs
    Co3_true_test = torch.tensor(CO3_t_test[[f'O3_{i}' for i in range(2,2+num_steps)]].values, dtype=torch.float32)

    with torch.no_grad():
        Co3_test_pred_flat,_ = pinn(X_test_expanded,T_test_flattened)
        Co3_test_pred = Co3_test_pred_flat.reshape(num_samples_test, num_steps_test).numpy()

    print("Predicted Ozone Matrix shape:", Co3_test_pred.shape)
    print("Predicted Ozone:", Co3_test_pred)

    # --- CT Calculation ---
    # Time intervals (dt)
    T_test_values = T_test[t_cols].values
    T_diffs = np.zeros_like(T_test_values)
    T_diffs[:, 1:] = np.diff(T_test_values, axis=1) # Time gap between steps
    CT_calc = Co3_test_pred * T_diffs
    y_pred = CT_calc.sum(axis=1)
    return pinn,y_pred

def bestmodels_ensemble_predict(models, X_train, y_train, X_test, method, n_splits=5):
    """
    Ensemble prediction with K-Fold cross-validation to determine weights or meta-model.
    Supports:
    - weighted (equal weights)
    - optimised (minimize MSE on OOF predictions)
    - stacking (train meta-learner on OOF predictions)
    """

    # --- Helper to get predictions from each base model ---
    def get_preds(model, features, X):
        """Safe prediction function that aligns features and avoids order mismatch errors."""
        if hasattr(model, "feature_names_in_"):
            features = list(model.feature_names_in_)
        X_sub = X.reindex(columns=features, fill_value=0)

        try:
            return model.predict(X_sub)
        except ValueError:
            return model.predict(X_sub.values)

    n_models = len(models)
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

    # === Build Out-of-Fold (OOF) predictions on TRAIN ===
    oof_preds = np.zeros((len(X_train), n_models))
    for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        for m_idx, (model, features) in enumerate(models):
            # clone model to avoid overwriting original
            model_clone = model.__class__(**model.get_params())
            model_clone.fit(X_tr[features], y_tr)
            oof_preds[val_idx, m_idx] = get_preds(model_clone, features, X_val)

    # === Predictions on TEST set ===
    test_preds = np.zeros((len(X_test), n_models))
    for m_idx, (model, features) in enumerate(models):
        test_preds[:, m_idx] = get_preds(model, features, X_test)

    # === Ensembling Strategies ===
    if method == "weighted":
        # Simple equal weights
        weights = np.ones(n_models) / n_models
        y_pred_test = np.dot(test_preds, weights)
        fitted_models = []
        for m_idx, (model, features) in enumerate(models):
            model_clone = model.__class__(**model.get_params())
            model_clone.fit(X_train[features], y_train)
            fitted_models.append((model_clone, features))
        ensemble_info = {"method": "weighted", "weights": weights,"models": fitted_models}

    elif method == "rmse_weighted":

        rmse_scores = np.array([
            np.sqrt(mean_squared_error(y_train, oof_preds[:, i]))
            for i in range(n_models)
        ])

        # Prevent division by zero
        rmse_scores = np.clip(rmse_scores, 1e-10, None)

        inv_rmse = 1 / rmse_scores
        weights = inv_rmse / np.sum(inv_rmse)
        y_pred_test = np.dot(test_preds, weights)
        fitted_models = []
        for m_idx, (model, features) in enumerate(models):
            model_clone = model.__class__(**model.get_params())
            model_clone.fit(X_train[features], y_train)
            fitted_models.append((model_clone, features))
        ensemble_info = {
            "method": "rmse_weighted",
            "weights": weights,
            "rmse_scores": rmse_scores,
            "models": fitted_models
        }

    elif method == "optimised":
        # Optimize weights on OOF predictions to minimize MSE
        def objective(weights):
            weights = np.array(weights) / np.sum(weights)
            y_pred_oof = np.dot(oof_preds, weights)
            return mean_squared_error(y_train, y_pred_oof)

        init_w = np.ones(n_models) / n_models
        bounds = [(0, 1)] * n_models
        constraints = ({'type': 'eq', 'fun': lambda w: np.sum(w) - 1})

        result = minimize(objective, init_w, bounds=bounds, constraints=constraints)
        opt_weights = result.x / np.sum(result.x)
        y_pred_test = np.dot(test_preds, opt_weights)
        fitted_models = []
        for m_idx, (model, features) in enumerate(models):
            model_clone = model.__class__(**model.get_params())
            model_clone.fit(X_train[features], y_train)
            fitted_models.append((model_clone, features))
        ensemble_info = {"method": "weighted", "weights": opt_weights,"models": fitted_models}

    elif method == "multi_optimised":
        def composite_objective(weights, oof_preds, y_true):
            weights = np.array(weights) / np.sum(weights)
            y_pred = np.dot(oof_preds, weights)
            mse = mean_squared_error(y_true, y_pred)
            mae = mean_absolute_error(y_true, y_pred)
            r2 = r2_score(y_true, y_pred)
            residuals = y_true - y_pred
            std_residuals = np.std(residuals)
            objective = 0.2*mse + 0.3*mae + 0.1*abs(std_residuals - np.mean(abs(residuals))) - 0.4*r2
            return objective

        init_w = np.ones(n_models) / n_models
        bounds = [(0, 1)] * n_models
        constraints = {'type': 'eq', 'fun': lambda w: np.sum(w) - 1}

        result = minimize(composite_objective, init_w,args=(oof_preds, y_train), bounds=bounds, constraints=constraints)
        opt_weights = result.x / np.sum(result.x)
        y_pred_test = np.dot(test_preds, opt_weights)
        fitted_models = []
        for m_idx, (model, features) in enumerate(models):
            model_clone = model.__class__(**model.get_params())
            model_clone.fit(X_train[features], y_train)
            fitted_models.append((model_clone, features))
        ensemble_info = {"method": "multi_optimised", "weights": opt_weights,"models": fitted_models}

    elif method == "stacking":
        # Train meta-learner on OOF predictions
        meta = XGBRegressor(n_estimators=200, learning_rate=0.01, random_state=42)
        meta.fit(oof_preds, y_train)
        y_pred_test = meta.predict(test_preds)
        fitted_models = []
        for m_idx, (model, features) in enumerate(models):
            model_clone = model.__class__(**model.get_params())
            model_clone.fit(X_train[features], y_train)
            fitted_models.append((model_clone, features))
        ensemble_info = {"method": "stacking", 'meta_model': meta,"models": fitted_models}

    elif method == "bagging":
        n_bags = int(input("Enter the number of bags for bagging: "))
        bag_preds = np.zeros((len(X_test), n_bags * n_models))
        bag_idx = 0

        for m_idx, (model, features) in enumerate(models):
            for b in range(n_bags):
                X_bag, y_bag = resample(X_train[features], y_train, replace=True, random_state=42 + b)
                model_clone = model.__class__(**model.get_params())
                model_clone.fit(X_bag, y_bag)
                bag_preds[:, bag_idx] = get_preds(model_clone, features, X_test)
                bag_idx += 1
        y_pred_test = np.mean(bag_preds, axis=1)
        fitted_models = []
        for m_idx, (model, features) in enumerate(models):
            model_clone = model.__class__(**model.get_params())
            model_clone.fit(X_train[features], y_train)
            fitted_models.append((model_clone, features))
        ensemble_info = {"method": "bagging", 'n_bags': n_bags,"models": fitted_models}

    else:
        raise ValueError("method must be one of: 'weighted', 'optimised', 'multi-optimised', 'stacking','bagging'")

    # Save the model
    save_dir="/content/drive/My Drive/Colab Notebooks/Ozonation model/"
    model_name=f"{method}ensemble_model.pkl"
    save_path=os.path.join(save_dir, model_name)
    joblib.dump(ensemble_info, save_path)
    print(f"✅ Ensemble model saved at: {save_path}")
    return y_pred_test,ensemble_info

def plot_cross_correlation(Ozoninput):

    # Display available columns
    print("Available columns in the dataset:")
    for i, column in enumerate(Ozoninput.columns, 1):
        print(f"{i}. {column}")

    # Allow user to select input features by number
    input_indices = input("Enter the numbers of the input features (comma-separated): ").strip().split(",")
    input_features = [Ozoninput.columns[int(i) - 1] for i in input_indices]

    # Allow user to select the target variable by number
    target_index = input("Enter the number of the target variable: ").strip()
    target_variable = Ozoninput.columns[int(target_index) - 1]

    # Prepare data
    X = Ozoninput[input_features]
    y = Ozoninput[target_variable]

    # Convert target variable to DataFrame if it's a Series
    #if isinstance(target_variable, pd.Series):
    #    target_variable = target_variable.to_frame()

    models_info = {}
    # Extract target column name
    #target_column = target_variable.columns[0]

    # set the feature importance to positive to use the top-8 most important issues.
    def get_top_n_nonzero(series_like, n=8):
        """Helper to get top n features with non-zero absolute values."""
        if isinstance(series_like, pd.DataFrame):
            series_like = series_like.iloc[:, 0]
        nonzero = series_like[series_like.abs() > 0]
        return nonzero.abs().sort_values(ascending=False).head(n).index.tolist()

    # Merge data and target_variable
    data = pd.concat([X, y], axis=1)

    # Pearson correlation
    corr_matrix = data.corr()
    pearson_series = corr_matrix[target_variable].drop(target_variable)
    top_features_pearson = get_top_n_nonzero(pearson_series)
    plt.figure(figsize=(14, 12))
    sns.heatmap(corr_matrix, annot=True, cmap="coolwarm", vmin=-1, vmax=1, linewidths=0.5)
    plt.title(f'Cross-Correlation Heatmap with {target_variable}')
    plt.show()
    # Correlation analysis
    print("\nPearson Correlation:")
    print(data.corr()[target_variable].sort_values(ascending=False))
    models_info["Pearson"] = top_features_pearson

    # Spearman Correlation
    spearman_corr = data.corr(method='spearman')
    spearman_series = spearman_corr[target_variable].drop(target_variable)
    top_features_spearman = get_top_n_nonzero(spearman_series)
    plt.figure(figsize=(14,12))
    sns.heatmap(spearman_corr, annot=True, cmap="YlGnBu", vmin=-1, vmax=1,linewidths=0.5)
    plt.title("Spearman Correlation Heatmap")
    plt.show()
    print("\nSpearman Correlation:")
    print(data.corr(method='spearman')[target_variable].sort_values(ascending=False))
    models_info["Spearman"] = top_features_spearman

    # Greedy Feature Selection (Sequential Forward Selection)
    print("\nGreedy Feature Selection (SFS with Linear Regression):")
    n_sfs = min(8, max(1, X.shape[1] - 1))
    sfs_model = LinearRegression()
    sfs = SequentialFeatureSelector(sfs_model, n_features_to_select=n_sfs, direction="forward")
    sfs.fit(X, y)
    selected_features_sfs = list(X.columns[sfs.get_support()])
    print("Selected Features:", selected_features_sfs)
    models_info["Greedy"] = selected_features_sfs

    # Mutual Information
    print("\nMutual Information:")
    mi_func = mutual_info_regression
    mi = mi_func(X, y)
    mi_series = pd.Series(mi, index=X.columns).sort_values(ascending=False)
    print(mi_series)
    top_features_mi = get_top_n_nonzero(mi_series)
    models_info["Mutual Information"] = top_features_mi

    # L1 Regularization (LassoCV)
    print("\nL1 Regularization (LassoCV):")
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    lasso = LassoCV(cv=5).fit(X_scaled, y)
    lasso_importance = pd.Series(np.abs(lasso.coef_), index=X.columns)
    print(lasso_importance.sort_values(ascending=False))
    top_features_l1 = get_top_n_nonzero(lasso_importance)
    models_info["L1 Regularization"] = top_features_l1

    # Permutation Importance (RandomForest)
    print("\nPermutation Importance (RandomForest):")
    rf = RandomForestRegressor(n_estimators=100, random_state=42)
    rf.fit(X, y)
    result = permutation_importance(rf, X, y, n_repeats=10, random_state=42)
    perm_importance = pd.Series(result.importances_mean, index=X.columns)
    print(perm_importance.sort_values(ascending=False))
    top_features_perm = get_top_n_nonzero(perm_importance)
    models_info["Permutation Importance"] = top_features_perm

    return X, y, input_features, target_variable,models_info


def kling_gupta_efficiency(y_test, y_pred):
    """Computes the Kling-Gupta Efficiency (KGE) metric."""
    y_true = np.asarray(y_test).flatten()
    y_predd = np.asarray(y_pred).flatten()
    eps = 1e-10   # numerical stability

    mean_obs = np.mean(y_true)
    mean_pred = np.mean(y_predd)

    std_obs = np.std(y_true)
    std_pred = np.std(y_predd)

    # Pearson correlation
    if std_obs < eps or std_pred < eps:
        r = 0.0
    else:
        r = np.corrcoef(y_true, y_predd)[0, 1]

    beta = mean_pred / (mean_obs +eps)  # Bias term

    cv_obs = std_obs / (mean_obs + eps)
    cv_pred = std_pred / (mean_pred + eps)
    gamma = cv_pred / (cv_obs + eps)

    kge = 1 - np.sqrt((r - 1)**2 + (beta - 1)**2 + (gamma - 1)**2)

    print("\nModel Performance:")
    print(f"Std obs: {std_obs:.4f}")
    print(f"Std pred: {std_pred:.4f}")
    print(f"R crosscor: {r:.4f}")
    print(f"Beta: {beta:.4f}")
    print(f"gamma: {gamma:.4f}")
    print(f"Kling - Gupta - Efficiency (KGE): {kge:.4f}")


    return kge

def plot_feature_importance(best_model, input_features,model_choice,X_train,results_dir):
    """
    Plot feature importance and SHAP values for Random Forest, XGBoost,
    and ensemble models.

    Parameters:
    - best_model: trained model or list of trained models
    - input_features: list of feature names
    - model_choice: integer from 1 to 5
    - X: input DataFrame or array (used for SHAP)
    """

    os.makedirs(results_dir, exist_ok=True)
    def plot_importance_bar(importances, title):
        importance_df = pd.DataFrame({"Feature": input_features, "Importance": importances})
        importance_df = importance_df.sort_values(by="Importance", ascending=False)
        plt.figure(figsize=(10, 6))
        plt.barh(importance_df["Feature"], importance_df["Importance"], color='skyblue')
        plt.xlabel("Feature Importance Score")
        plt.ylabel("Features")
        plt.title(title)
        plt.gca().invert_yaxis()
        plt.savefig(os.path.join(results_dir, "Feauture importance.png"))
        plt.show()

    def plot_shap_summary(model, X_data, title):


        try:
            explainer = shap.TreeExplainer(model)
            shap_values = explainer.shap_values(X_data)
            if isinstance(shap_values, list):
                shap_values = shap_values[1] if len(shap_values) > 1 else shap_values[0]
            plt.figure()
            plt.title(title)
            shap.summary_plot(shap_values, X_data, show=False, max_display=10)
            plt.show()

        except Exception as e:

            print(f"⚠️ TreeExplainer failed: {e}")
            print("⚠️ Falling back to KernelExplainer (slow)...")

            try:
                background = shap.sample(X_data, 20)

                explainer = shap.KernelExplainer(model.predict, background)
                shap_values = explainer.shap_values(shap.sample(X_data, 100))

                shap.summary_plot(shap_values, shap.sample(X_data, 100))

            except Exception as e2:
                print(f"❌ SHAP completely failed: {e2}")

    if model_choice == 1:  # Random Forest
        print("Random Forest - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "Random Forest - Feature Importance")
        print("Random Forest - SHAP Summary")
        plot_shap_summary(best_model, X_train, "Random Forest - SHAP Summary")

    elif model_choice == 2:  # XGBoost
        print("XGBoost - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "XGBoost - Feature Importance")
        print("XGBoost - SHAP Summary")
        plot_shap_summary(best_model, X_train, "XGBoost - SHAP Summary")

    elif model_choice == 9:  # GBoost
        print("GBoost - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "GBoost - Feature Importance")
        print("GBoost - SHAP Summary")
        plot_shap_summary(best_model, X_train, "GBoost - SHAP Summary")

    elif model_choice == 10:  # AdaBoost
        print("AdaBoost - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "AdaBoost - Feature Importance")
        print("AdaBoost - SHAP Summary")
        background = shap.sample(X_train, 50).values
        X_sample = shap.sample(X_train, 50).values

        explainer = shap.KernelExplainer(best_model.predict, background)

        shap_values = explainer.shap_values(X_sample)

        shap.summary_plot(shap_values, X_sample, feature_names=input_features)

    elif model_choice == 11:  # LGBoost
        print("LGBoost - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "LGBoost - Feature Importance")
        print("LGBoost - SHAP Summary")
        plot_shap_summary(best_model, X_train, "LGBoost - SHAP Summary")

    elif model_choice == 14:  # XTC
        print("XTC - Gradient Boost hybrid - Feature Importance")
        plot_importance_bar(best_model.feature_importances_, "XTC - Gradient Boost - Feature Importance")
        print("XTC - Gradient Boost hybrid - SHAP Summary")
        plot_shap_summary(best_model, X_train, "Gradient Boost hybrid - SHAP Summary")

    else:
        raise ValueError("Feature importance is only supported for model choices 1 to 5.")

def predict_ozone_exposure(X, y, input_features, target_variable, k_UVA,CO3_t,Ko3,T, models_info,voldiff):
    """
    Main function to predict ozone exposure (CT) using user-selected model and inputs.
    """

    # Allow user to select a model
    print("\nSelect a model:")
    print("1. Random Forest")
    print("2. XGBoost")
    print("3. Multi ensemble Random Forest")
    print("4. Multi ensemble XGBoost")
    print("5. Random Forest - XGBoost combination")
    print("6. Random Forest - Combination of 8 top-8 most important features (as per feature importance methods)")
    print("7. XGBoost - Combination of 8 top-8 most important features (as per feature importance methods)")
    print("8. Random Forest & XGBoost - Combination of 8 top-8 most important features (as per feature importance methods)")
    print("9. Gradient Boosting")
    print("10. AdaBoost")
    print("11. LightGBM")
    print("12. Random Forrest - XGBoost - GBoost ensemble of up to 18 models")
    print("13. Physics-Informed MLP")
    print("14. XTC - hybrid model")
    print("15. Ensemble of Best Models - Weighted")
    try:
        model_choice = int(input("Enter the number corresponding to your choice:"))
    except ValueError:
        print("Invalid choice. Exiting.")
        return

    # Train-test split
    testsize = float(input("Set the testing size(e.g. 0.1 = 10% for testing):"))
    rdmstate = int(input("Set the random state:"))
    X_train, X_test, y_train, y_test,idx_train, idx_test = train_test_split(X, y,range(len(X)), test_size=testsize, random_state=rdmstate) # this is for the decision trees approach
    X_train = pd.DataFrame(X_train, columns=input_features)
    X_test = pd.DataFrame(X_test, columns=input_features)
    feature_scaler = StandardScaler()
    target_scaler = StandardScaler()
    X_train_scaled = feature_scaler.fit_transform(X_train) # this is for the decision trees approach
    y_train_scaled = target_scaler.fit_transform(y_train.values.reshape(-1, 1))
    scaler_dir = input('Set the folder name:').strip()
    save_dir="/content/drive/My Drive/Colab Notebooks/Ozonation model/"
    full_save_dir = os.path.join(save_dir, scaler_dir)
    os.makedirs(full_save_dir, exist_ok=True)
    model_name1=f"{model_choice}feature_scaler.pkl"
    model_name2=f"{model_choice}target_scaler.pkl"
    save_path1=os.path.join(full_save_dir, model_name1)
    save_path2=os.path.join(full_save_dir, model_name2)
    X_test_scaled = feature_scaler.transform(X_test)
    joblib.dump(feature_scaler, save_path1)
    joblib.dump(target_scaler, save_path2)
    #X_train_ps, X_test_ps, y_train_ps, y_test_ps,idx_train, idx_test = train_test_split(X, y,range(len(X)), test_size=testsize, random_state=42)# this is for the PINN approach
    CO3_t_train=CO3_t.iloc[idx_train]
    CO3_t_test=CO3_t.iloc[idx_test]
    T_train=T.iloc[idx_train]
    T_test=T.iloc[idx_test]

    # Model selection
    best_model = None

    if model_choice == 1:
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        best_model = random_forest_model(X_train_scaled, y_train_scaled)
        y_pred_scaled = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 2:
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        best_model = xgboost_model(X_train_scaled, y_train_scaled)
        y_pred_scaled = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 3:
        y_train = y_train.ravel() # Reshape y_train before fitting
        y_pred_rf, best_models, best_scores = multi_random_forest_model(X_train, y_train, input_features, X_test)
        y_pred = target_scaler.inverse_transform(y_pred_rf.reshape(-1, 1))
        y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))
        best_model = best_models

    elif model_choice == 4:
        y_train = y_train.ravel() # Reshape y_train before fitting
        y_pred_xgb, best_models, best_scores = multi_xgboost_model(X_train, y_train, input_features, X_test)
        y_pred = target_scaler.inverse_transform(y_pred_xgb.reshape(-1, 1))
        y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))
        best_model = best_models

    elif model_choice == 5:
        y_train = y_train.ravel() # Reshape y_train before fitting
        y_pred_rf, best_models_rf, best_scores_rf = multi_random_forest_model(X_train, y_train, input_features, X_test)
        y_pred_xgb, best_models_xgb, best_scores_xgb = multi_xgboost_model(X_train, y_train, input_features, X_test)
        y_pred_final,combined_models = combined_rf_xgb_ensemble(best_models_rf, best_scores_rf, best_models_xgb, best_scores_xgb, X_test)
        # Combine both models' predictions equally (can be weighted if needed)
        y_pred = target_scaler.inverse_transform(y_pred_final.reshape(-1, 1))
        y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))
        best_model = combined_models

    elif model_choice == 6:
        y_train = y_train.ravel() # Reshape y_train before fitting
        best_models, y_pred_final = top_features_random_forest_model(X_train, y_train, input_features, X_test, y_test,models_info)
        y_pred = target_scaler.inverse_transform(y_pred_final .reshape(-1, 1))
        y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))
        best_model = best_models

    elif model_choice == 7:
        y_train = y_train.ravel() # Reshape y_train before fitting
        best_models, y_pred_final = top_features_xgb_model(X_train, y_train, input_features, X_test, y_test,models_info)
        y_pred = target_scaler.inverse_transform(y_pred_final .reshape(-1, 1))
        y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))
        best_model = best_models

    elif model_choice == 8:
        y_train = y_train.ravel() # Reshape y_train before fitting
        y_pred_rf, best_models_rf, best_scores_rf = multi_random_forest_model(X_train, y_train, input_features, X_test)
        y_pred_xgb, best_models_xgb, best_scores_xgb = multi_xgboost_model(X_train, y_train, input_features, X_test)
        y_pred_final,combined_models = combined_rf_xgb_ensemble(best_models_rf, best_scores_rf, best_models_xgb, best_scores_xgb, X_test)
        # Combine both models' predictions equally (can be weighted if needed)
        y_pred = target_scaler.inverse_transform(y_pred_final.reshape(-1, 1))
        y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))
        best_model = combined_models

    elif model_choice == 9:
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        best_model = gb_model(X_train_scaled, y_train_scaled)
        y_pred_scaled = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 10:
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        best_model = adaboost_model(X_train_scaled, y_train_scaled)
        y_pred_scaled = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 11:
        y_train_scaled = y_train_scaled.ravel() # Reshape y_train before fitting
        best_model = lgb_model(X_train_scaled, y_train_scaled)
        y_pred_scaled = best_model.predict(X_test_scaled)
        y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 12:
        y_train = y_train.ravel() # Reshape y_train before fitting
        print("\nSelect ensemble methodology:")
        print("1. Weighted Average")
        print("2. Optimised Weights")
        print("3. Stacking")
        try:
           ens_method = int(input("Enter the number corresponding to your choice:"))
        except ValueError:
            print("Invalid choice. Exiting.")
            return
        if ens_method == 1 :
            method = "weighted"
            print(f"selected method is{method}")
        elif ens_method == 2 :
            method = "optimised"
            print(f"selected method is{method}")
        elif ens_method == 3 :
            method = 'stacking'
            print(f"selected method is{method}")
        else:
            print("Invalid choice. Exiting.")
            return
        method
        y_pred, best_models, scores = run_multi_model_ensemble(X_train, y_train, X_test, ens_method, k=10,random_state=42)

        y_pred = target_scaler.inverse_transform(y_pred_rf.reshape(-1, 1))
        y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))
        best_model = best_models

    elif model_choice == 13:

        pinn,y_pred = physics_informed_mlp(X_train, X_test, y_train, Ko3, k_UVA, CO3_t_train, CO3_t_test,T_train,T_test)
        best_model = pinn
#        y_pred = y_pred.to_numpy()
        y_pred = y_pred.reshape(-1, 1)
        #y_pred = target_scaler.inverse_transform(y_pred)
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))
        #pinn, Ko3.item(), Kuva.item()

    elif model_choice == 14:
        %run "/content/drive/My Drive/Colab Notebooks/Ozonation model/X-TFCmodel.ipynb"
        best_model, CT_pred_xtfc, CT_pred_hybrid, Xtr_hybrid, Xte_hybrid,k_train,hybrid_features= X_TFC_CT_compartment_model(X_train, X_test, y_train, y_test,T,voldiff)
#       y_pred = y_pred.to_numpy()
        y_pred = CT_pred_hybrid
        #y_pred = target_scaler.inverse_transform(y_pred)
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))

    elif model_choice == 15:
        from pathlib import Path
        load_dir = Path("/content/drive/My Drive/Colab Notebooks/Ozonation model/OZONE BEST MODEL TRAINED MODELS/New dataset/Mix features")
        files = sorted([f for f in os.listdir(load_dir) if f.endswith(".pkl")])
        print("\nAvailable models:")
        for i, f in enumerate(files):
            print(f"{i}: {f}")

        n_models = int(input("\nHow many models do you want to combine? "))
        FEATURE_MAP = {
             "XGB_test_7_v2_42.pkl": [
             "month","Chamber","WaterTemperature",
             "RFTurbidity","FlowRate","Ozonedosage","Residencetime"
             ],
             "best_xgb_model_6features_newdataset.pkl": [
             "hour","Chamber","WaterTemperature",
             "RFTurbidity","Ozonedosage","FlowRate"
             ],
             "xgb_model_5featuresnohour_newdataset.pkl": [
             "Chamber","WaterTemperature",
             "RFTurbidity","Ozonedosage","FlowRate"
             ],
             "XGB_test_5_v2_42.pkl": [
             "Chamber","WaterTemperature",
             "RFTurbidity","Ozonedosage","FlowRate"
             ],
             "gb_model_6features_newdataset.pkl": [
             "hour","Chamber","WaterTemperature",
             "RFTurbidity","FlowRate","Ozonedosage"
             ],
             "best_gb_model_7features_newdataset.pkl": [
             "hour", "Chamber","WaterTemperature",
             "RFTurbidity","FlowRate","Ozonedosage","Residencetime"
             ],
             "GB_test_7_v2_42.pkl": [
             "Chamber","WaterTemperature","month",
             "RFTurbidity","FlowRate","Ozonedosage","Residencetime"
             ],
             "gb_model_5featuresnohour_newdataset.pkl": [
             "Chamber","WaterTemperature",
             "RFTurbidity","FlowRate","Ozonedosage"
             ],
              "GB_test_5_v2_42.pkl": [
             "Chamber","WaterTemperature",
             "RFTurbidity","FlowRate","Ozonedosage"
             ],
             "best_rf_model_7features_newdataset.pkl": [
             "hour","Chamber","WaterTemperature",
             "RFTurbidity","FlowRate","Ozonedosage","Residencetime"
             ],
             "rf_model_6featuresnohour_newdataset.pkl": [
             "Chamber","WaterTemperature",
             "RFTurbidity","FlowRate","Ozonedosage"
             ],
        }
        models = []

        for i in range(n_models):
            idx = int(input(f"Select model {i+1} index: "))
            fname = files[idx]

            model_path = load_dir / fname
            model = joblib.load(model_path)

            if fname not in FEATURE_MAP:
                raise ValueError(f"No feature list defined for {fname}")

            features = FEATURE_MAP[fname]
            models.append((model, features))

            print(f"Loaded: {fname}")
        print("\nSelect ensemble methodology:")
        print("1. Average")
        print("2. RMSE Average")
        print("3. Optimised Weights")
        print("4. Multi-Optimised Weights")
        print("5. Stacking")
        print("6. Bagging")
        try:
           ens_method = int(input("Enter the number corresponding to your choice:"))
        except ValueError:
            print("Invalid choice. Exiting.")
            return
        if ens_method == 1 :
            method = "weighted"
        elif ens_method == 2 :
            method = "rmse_weighted"
        elif ens_method == 3 :
            method = "optimised"
        elif ens_method == 4 :
            method = "multi_optimised"
        elif ens_method == 5 :
            method = 'stacking'
        elif ens_method == 6 :
            method = 'bagging'
        else:
            print("Invalid choice. Exiting.")
            return
        #X_train_split, X_valid, y_train_split, y_valid = train_test_split(X_train, y_train, test_size=0.1, random_state=42)
        X_train_scaled = pd.DataFrame(X_train_scaled,columns=input_features,index=X_train.index)
        X_test_scaled = pd.DataFrame(X_test_scaled,columns=input_features,index=X_test.index)
        y_train_scaled = pd.Series(y_train_scaled.ravel(),index=y_train.index)
        y_pred_scaled,ensemble_info = bestmodels_ensemble_predict(models, X_train_scaled, y_train_scaled, X_test_scaled,method,n_splits=5)
        #y_pred = bestmodels_ensemble_predict(models, X_test, method=method,X_valid=X_valid,y_valid=y_valid)
        y_pred = target_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1))
        #y_test = target_scaler.inverse_transform(y_test.reshape(-1, 1))
        save_dir="/content/drive/My Drive/Colab Notebooks/Ozonation model/OZONE BEST MODEL TRAINED MODELS/New dataset/Mix features"
        model_name1="feature_scaler.pkl"
        model_name2="target_scaler.pkl"
        save_path1=os.path.join(save_dir, model_name1)
        save_path2=os.path.join(save_dir, model_name2)
        joblib.dump(feature_scaler, save_path1)
        joblib.dump(target_scaler, save_path2)
    else:
        print("Invalid choice. Exiting.")
        return None

    # Calculate performance metrics
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    si = rmse/np.mean(y_test)
    kge = 0
    #kge = kling_gupta_efficiency(y_test, y_pred)


    print("\nModel Performance:")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"Scatter Index: {si:.4f}")
    print(f"R²: {r2:.4f}")
    print(f"Kling - Gupta - Efficiency (KGE): {kge:.4f}")

    # Create results directory
    results_dir = input('Set the folder name:')
    os.makedirs(results_dir, exist_ok=True)

    # Save performance metrics to a txt file
    metrics_file = os.path.join(results_dir, "performance_metrics.txt")
    with open(metrics_file, "w") as f:
        f.write("Model Performance:\n")
        f.write(f"MAE: {mae:.4f}\n")
        f.write(f"MSE: {mse:.4f}\n")
        f.write(f"RMSE: {rmse:.4f}\n")
        f.write(f"Scatter Index: {si:.4f}\n")
        f.write(f"R²: {r2:.4f}\n")
        f.write(f"Kling - Gupta - Efficiency (KGE): {kge:.4f}\n")
        if model_choice == 15:
               f.write("Ensemble Information\n")
               f.write("====================\n\n")
               f.write(f"method: {ensemble_info['method']}\n\n")
               if "weights" in ensemble_info:
                  f.write("Weights:\n")
                  for i, w in enumerate(ensemble_info["weights"]):
                      f.write(f"  Model {i+1}: {w:.6f}\n")
                  f.write("\n")
               if "models" in ensemble_info:
                  f.write("Base Models Used:\n")
                  for i, (model_obj, features) in enumerate(ensemble_info["models"]):
                      f.write(f"  Model {i+1}: {type(model_obj).__name__}\n")
                      f.write(f"    Features: {', '.join(features)}\n")
                  f.write("\n")
               if "rmse_scores" in ensemble_info:
                      f.write("RMSE Scores:\n")
                      for i, rmse in enumerate(ensemble_info["rmse_scores"]):
                          f.write(f"  Model {i+1}: {rmse:.6f}\n")
                      f.write("\n")
    # Save input features if available
    try:
        feature_file = os.path.join(results_dir, f"{X.shape[1]} used_features.txt")
        with open(feature_file, "w") as f:
            f.write(f"Number of Features: {X.shape[1]}\n")
            f.write("Input Features Used:\n")
            f.write("\n".join(X_test.columns))
    except:
        pass  # Skip if X_test doesn't have .columns

    # Save predicted vs observed plot
    plt.figure(figsize=(6, 6))
    plt.scatter(y_test, y_pred, color='blue', alpha=0.7, label='Predictions')
    plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', label='1:1 Line')
    plt.xlabel("Observed CT")
    plt.ylabel("Predicted CT")
    plt.title("Predicted vs Observed CT")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(results_dir, "predicted_vs_observed.png"))
    plt.show()

    # Plot predicted vs observed data
    #plt.scatter(y_test, y_pred, color='blue')
    #plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], color='red', linestyle='--')
    #plt.xlabel("Observed CT")
    #plt.ylabel("Predicted CT")
    #plt.title("Predicted vs Observed CT")
    #plt.show()

    # Plot feature importance if model is Random Forest or XGBoost
    if model_choice in [1, 2, 9, 10, 11]:
        plot_feature_importance(best_model, input_features,model_choice,X_train,results_dir)

    return best_model, mae, mse, rmse, r2, si, kge, y_test, y_pred